<a href="https://colab.research.google.com/github/AzizulHakim00/Glaucomma/blob/main/Glaucomma_RimGraphDG_V44_Single_Cell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RimGraph-DG V4.4 — verified Colab launcher
Select a **T4 GPU** runtime, then run the single code cell below. The launcher performs a real full-model forward/backward preflight before the long experiment and refuses to report success unless the Drive completion marker exists.

In [ ]:
# RimGraph-DG V4.4 — T4-safe, visible progress, verified completion.
GLAUCOMMA_OVERRIDES = {
    'manual_data_dir': '',
    'fast_dev_run': False,
    'run_name': 'paper_run_v44',
    'code_revision': 'rimgraph-dg-v4.4-20260808',
    'seeds': [2029],
    'run_global_baseline': True,
    'run_full_model': True,
    'run_optuna': False,
    'optuna_trials': 8,
    'image_size': 320,
    'batch_size': 2,
    'grad_accum': 2,
    'fpn_dim': 128,
    'gradient_checkpointing': True,
    'baseline_epochs': 12,
    'full_epochs': 30,
    'seg_warmup_epochs': 5,
    'anatomy_warmup_epochs': 10,
    'num_workers': 0,
    'n_visual_examples': 2,
}

import hashlib
import json
import traceback
import urllib.request
from pathlib import Path

import torch
print('=== LAUNCHER GPU CHECK ===', flush=True)
print('PyTorch:', torch.__version__, flush=True)
print('CUDA available:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('T4 GPU is not active. Colab: Runtime > Change runtime type > T4 GPU, reconnect, then rerun.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print('==========================', flush=True)

COMMIT = '9331fb269ac392e8449889769aaf0c54e510f8be'
EXPECTED_RAW_SHA256 = '46ba27c7446662460456bc2bab186729c0df1b3e76533ce44f208150208335e2'
ROOT = f'https://raw.githubusercontent.com/AzizulHakim00/Glaucomma/{COMMIT}'
parts = [f'v4_parts/part_{i:02d}.py' for i in range(7)]
raw_code = '\n'.join(urllib.request.urlopen(f'{ROOT}/{name}').read().decode('utf-8') for name in parts)
actual_raw = hashlib.sha256(raw_code.encode('utf-8')).hexdigest()
assert actual_raw == EXPECTED_RAW_SHA256, f'V4 raw runner integrity check failed: {actual_raw}'

patch_specs = [
    ('runner_patch_v41.py', 'apply_v41'),
    ('runner_patch_v42.py', 'apply_v42'),
    ('runner_patch_v43.py', 'apply_v43'),
    ('runner_patch_v43_autograd.py', 'apply_v43_autograd'),
    ('runner_patch_v44_runtime.py', 'apply_v44_runtime'),
]
code = raw_code
for patch_name, function_name in patch_specs:
    print(f'[LAUNCHER] applying {patch_name}', flush=True)
    source = urllib.request.urlopen(f'{ROOT}/{patch_name}').read().decode('utf-8')
    namespace = {}
    exec(compile(source, patch_name, 'exec'), namespace, namespace)
    code = namespace[function_name](code)
compile(code, 'rimgraph_dg_v44_single_cell.py', 'exec')
print('[LAUNCHER] code assembly PASSED', flush=True)

try:
    exec(code, globals(), globals())
    completion = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v44/RUN_COMPLETED.json')
    if not completion.exists():
        raise RuntimeError('Runner returned without RUN_COMPLETED.json. This is treated as a FAILED run, not success.')
    print('\n✅ V4.4 VERIFIED COMPLETION:', completion, flush=True)
except BaseException:
    trace = traceback.format_exc()
    print('\n=== V4.4 FAILURE TRACEBACK ===', flush=True)
    print(trace, flush=True)
    local_failure = Path('/content/RimGraph_V44_FAILURE_TRACEBACK.txt')
    try:
        local_failure.write_text(trace, encoding='utf-8')
    except Exception:
        pass
    try:
        failure_dir = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v44')
        failure_dir.mkdir(parents=True, exist_ok=True)
        (failure_dir / 'FAILURE_TRACEBACK.txt').write_text(trace, encoding='utf-8')
        (failure_dir / 'FAILURE_STATUS.json').write_text(json.dumps({'status': 'failed', 'traceback_file': str(failure_dir / 'FAILURE_TRACEBACK.txt')}, indent=2), encoding='utf-8')
    except Exception as drive_error:
        print(f'Could not persist failure to Drive: {drive_error}', flush=True)
    raise


=== LAUNCHER GPU CHECK ===
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
[LAUNCHER] applying runner_patch_v41.py
[LAUNCHER] applying runner_patch_v42.py
[LAUNCHER] applying runner_patch_v43.py
[LAUNCHER] applying runner_patch_v43_autograd.py
[LAUNCHER] applying runner_patch_v44_runtime.py
[LAUNCHER] code assembly PASSED
Mounted at /content/drive
Resolved Colab output: /content/Glaucomma_runs/paper_run_v44
Resolved Drive output: /content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v44
Drive write verification: PASSED

=== RIMGRAPH V4.4 RUNTIME ===
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB



## RimGraph-DG V4 configuration

,setting,value
0,kaggle_dataset,arnavjain1/glaucoma-datasets
1,manual_data_dir,
2,sources,"['ORIGA', 'REFUGE', 'G1020']"
3,fold_targets,"['ORIGA', 'REFUGE', 'G1020']"
4,image_size,320
5,num_workers,0
6,canonicalize_laterality,True
7,exclude_cross_source_duplicates,True
8,project_name,RimGraph_DG_V4
9,run_name,paper_run_v44


## Downloading or locating Kaggle dataset

100%|██████████| 5.55G/5.55G [01:03<00:00, 94.5MB/s]

...

model.safetensors: downloading bytes:           |  0.00B            

[BACKBONE] ready: convnext_tiny.fb_in22k_ft_in1k
[PREFLIGHT] full-model forward/backward PASSED | peak allocated=0.35 GB


## Seed 2029 — held-out ORIGA

[BACKBONE] loading convnext_tiny.fb_in22k_ft_in1k pretrained=True ...
[BACKBONE] ready: convnext_tiny.fb_in22k_ft_in1k
[BASELINE] epoch 1/12 start | batches=728
[BASELINE] epoch 1 batch 1/728
[BASELINE] epoch 1 batch 182/728
[BASELINE] epoch 1 batch 364/728
[BASELINE] epoch 1 batch 546/728
[BASELINE] epoch 1 batch 728/728
[BASELINE] epoch 1 done | loss=0.1999 auroc=0.7762 auprc=0.7883
[BASELINE] epoch 2/12 start | batches=728
[BASELINE] epoch 2 batch 1/728
[BASELINE] epoch 2 batch 182/728
[BASELINE] epoch 2 batch 364/728
[BASELINE] epoch 2 batch 546/728
[BASELINE] epoch 2 batch 728/728
[BASELINE] epoch 2 done | loss=0.1632 auroc=0.8722 auprc=0.8823
[BASELINE] epoch 3/12 start | batches=728
[BASELINE] epoch 3 batch 1/728
[BASELINE] epoch 3 batch 182/728
[BASELINE] epoch 3 batch 364/728
[BASELINE] epoch 3 batch 546/728
[BASELINE] epoch 3 batch 728/728
[BASELINE] epoch 3 done | loss=0.1647 auroc=0.9199 auprc=0.9180
[BASELINE] epoch 4/12 start | batches=728
[BASELINE] epoch 4 batch 1/728
[

[BACKBONE] ready: convnext_tiny.fb_in22k_ft_in1k
[RIMGRAPH] epoch 1/30 start | stage=global_seg | batches=728
[RIMGRAPH] epoch 1 stage=global_seg batch 1/728
[RIMGRAPH] epoch 1 stage=global_seg batch 182/728
[RIMGRAPH] epoch 1 stage=global_seg batch 364/728
[RIMGRAPH] epoch 1 stage=global_seg batch 546/728
[RIMGRAPH] epoch 1 stage=global_seg batch 728/728
[RIMGRAPH] epoch 1 done | stage=global_seg loss=0.2091 auroc=0.7652 auprc=0.7669 disc_dice=nan cup_dice=nan proto_active=n/a
[RIMGRAPH] epoch 2/30 start | stage=global_seg | batches=728
[RIMGRAPH] epoch 2 stage=global_seg batch 1/728
[RIMGRAPH] epoch 2 stage=global_seg batch 182/728
[RIMGRAPH] epoch 2 stage=global_seg batch 364/728
[RIMGRAPH] epoch 2 stage=global_seg batch 546/728
[RIMGRAPH] epoch 2 stage=global_seg batch 728/728
[RIMGRAPH] epoch 2 done | stage=global_seg loss=0.1691 auroc=0.8763 auprc=0.8807 disc_dice=nan cup_dice=nan proto_active=n/a
[RIMGRAPH] epoch 3/30 start | stage=global_seg | batches=728
[RIMGRAPH] epoch 3 sta